## 1. Setup: Navigate to parent directory and import dependencies

In [ ]:
%cd ..
from pathlib import Path
import json
import torch
import numpy as np
from transformers import AutoTokenizer

## 2. Import spesia_ner classes

In [ ]:
from spesia_ner.datasets import ClinicalRecordsDataset
from spesia_ner.data_models import Record
from spesia_ner.metrics import compute_metrics

## 3. Load the dummy dataset

The dummy dataset is located at `data/dummy_data/fake_data.jsonl`. It contains records with text and entity annotations (PERSON, ORGANIZATION, LOCATION).

In [ ]:
# Define the path to the dummy data directory
dummy_data_path = Path("data/dummy_data")
print(f"Dummy data directory: {dummy_data_path.resolve()}")
print(f"Files present: {list(dummy_data_path.glob('*'))}")

## 4. Create a ClinicalRecordsDataset with IO annotation scheme

The IO (Inside-Outside) scheme is simpler than BIO and is suitable for initial exploration.

In [ ]:
# Load the dummy dataset with IO annotation scheme (no split yet)
dataset_io = ClinicalRecordsDataset(
    path=dummy_data_path,
    tokenizer=None,  # We'll add tokenizer in next steps
    label_type="tags",  # We're labeling entity types as "tags"
    annotation_scheme="IO",  # IO scheme (Inside-Outside)
    split=None,  # Load all records without splitting
)

print(f"Total records loaded (IO scheme): {len(dataset_io.records)}")
print(f"Tags found: {sorted(dataset_io.tags)}")

## 5. Load dataset with splits (train/val/test)

Now let's load the dataset with automatic train/val/test splits using iterative stratification.

In [ ]:
# Load training split with stratification
train_dataset = ClinicalRecordsDataset(
    path=dummy_data_path,
    label_type="tags",
    annotation_scheme="IO",
    split="train",
    split_ratio={"train": 0.6, "val": 0.2, "test": 0.2},
    data_split_method="iterative",  # Use iterative stratification
    random_seed=42,
)

# Load validation split
val_dataset = ClinicalRecordsDataset(
    path=dummy_data_path,
    label_type="tags",
    annotation_scheme="IO",
    split="val",
    split_ratio={"train": 0.6, "val": 0.2, "test": 0.2},
    data_split_method="iterative",
    random_seed=42,
)

# Load test split
test_dataset = ClinicalRecordsDataset(
    path=dummy_data_path,
    label_type="tags",
    annotation_scheme="IO",
    split="test",
    split_ratio={"train": 0.6, "val": 0.2, "test": 0.2},
    data_split_method="iterative",
    random_seed=42,
)

print(f"Train set size: {len(train_dataset.records)}")
print(f"Val set size: {len(val_dataset.records)}")
print(f"Test set size: {len(test_dataset.records)}")
print(f"Total: {len(train_dataset.records) + len(val_dataset.records) + len(test_dataset.records)}")

## 6. Inspect a sample record

In [ ]:
# Get a sample record
sample_record = train_dataset.records[0]
print(f"Sample text: {sample_record.text[:150]}...")
print(f"\nTags: {sample_record.tags}")
print(f"\nEntity spans (label): {sample_record.entity_spans}")

## 7. Using BIO annotation scheme

Now let's load the same data with the BIO (Begin-Inside-Outside) annotation scheme, which is more complex but provides better boundary information.

In [ ]:
# Load training split with BIO annotation scheme
train_dataset_bio = ClinicalRecordsDataset(
    path=dummy_data_path,
    label_type="tags",
    annotation_scheme="BIO",  # BIO scheme (Begin-Inside-Outside)
    split="train",
    split_ratio={"train": 0.6, "val": 0.2, "test": 0.2},
    data_split_method="iterative",
    random_seed=42,
)

print(f"Train set size (BIO): {len(train_dataset_bio.records)}")
print(f"Tags with BIO scheme: {sorted(train_dataset_bio.tags)}")

## 8. Initialize a tokenizer for the dataset

In [ ]:
# Load a tokenizer (using a small model for demo)
model_id = "jhu-clsp/mmBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
print(f"Tokenizer loaded: {model_id}")
print(f"Vocab size: {tokenizer.vocab_size}")

## 9. Create a tokenized dataset

In [ ]:
# Create a dataset with tokenizer attached
train_tokenized = ClinicalRecordsDataset(
    path=dummy_data_path,
    tokenizer=tokenizer,
    label_type="tags",
    annotation_scheme="IO",
    split="train",
    split_ratio={"train": 0.6, "val": 0.2, "test": 0.2},
    data_split_method="iterative",
    max_length=512,
    random_seed=42,
)

print(f"Tokenized dataset size: {len(train_tokenized)}")
print(f"Number of labels: {train_tokenized.num_labels}")
print(f"Labels to consider: {train_tokenized.labels_to_consider}")

## 10. Access a tokenized sample from the dataset

In [ ]:
# Get a tokenized sample
sample_idx = 0
sample = train_tokenized[sample_idx]
print(f"Sample keys: {sample.keys()}")
print(f"\nInput shape: {sample['input_ids'].shape}")
print(f"Labels shape: {sample['labels'].shape}")
print(f"\nFirst few tokens: {tokenizer.decode(sample['input_ids'][:20])}")

## 11. Get label statistics

In [ ]:
# Get label count statistics
label_counts = train_dataset.get_label_count(label_type="tags")
print("Label distribution in training set:")
print(label_counts)
print(f"\nTotal labels: {label_counts.sum()}")

## 12. Export dataset to different formats

In [ ]:
# Export to JSONL format
output_path = Path("outputs_example")
output_path.mkdir(exist_ok=True)

# Export training data
train_dataset.export(
    output_path / "train_dummy.jsonl",
    format="jsonl",
)

print(f"Training data exported to: {output_path / 'train_dummy.jsonl'}")

# Check export file
with open(output_path / "train_dummy.jsonl", "r") as f:
    first_line = json.loads(f.readline())
    print(f"\nFirst exported record keys: {first_line.keys()}")

## 13. Summary

This notebook demonstrates:

1. **Loading data**: Using `ClinicalRecordsDataset` to load records from the dummy dataset
2. **Annotation schemes**: Both IO and BIO annotation schemes
3. **Data splitting**: Train/val/test splits with iterative stratification
4. **Tokenization**: Integration with HuggingFace transformers tokenizers
5. **Data access**: Accessing tokenized samples and label statistics
6. **Exporting**: Saving datasets to different formats

All these operations work with the bundled `fake_data.jsonl` file, making this notebook fully reproducible without requiring access to private clinical datasets.